# Customer Analytics — Employee Metrics

Runs on a Microsoft **Fabric Spark** notebook (`synapse_pyspark` kernel). Fabric
supplies the ambient `spark` session, so no `SparkSession` is constructed here.

Attach a Lakehouse before running if you extend this to persist results
(`.write.mode("overwrite").saveAsTable(...)` writes Delta by default in Fabric).

In [ ]:
from datetime import date

import pyspark.sql.functions as F
from pyspark.sql.types import (
    DateType,
    DoubleType,
    IntegerType,
    StringType,
    StructField,
    StructType,
)


def show(df, n=20):
    """Render with Fabric's rich display(); fall back to .show() off-platform."""
    renderer = globals().get("display")
    if callable(renderer):
        renderer(df.limit(n))
    else:
        df.show(n, truncate=False)


# Column order here is the contract the analytics cell below relies on.
employees_schema = StructType([
    StructField("EmployeeID", StringType(), False),
    StructField("HireDate", DateType(), False),
    StructField("Department", StringType(), True),
    StructField("Region", StringType(), True),
    StructField("JobTitle", StringType(), False),
    StructField("Category", StringType(), False),
    StructField("Salary", DoubleType(), False),
    StructField("YearsExperience", IntegerType(), False),
])

# Sample data — tuple order matches employees_schema (Department, then Region).
employees_data = [
    ("EMP001", date(2020, 3, 15),  "Engineering", "West",  "Data Engineer",     "Full-Time",  95000.00, 4),
    ("EMP002", date(2019, 7, 22),  "Marketing",   "East",  "Analyst",           "Full-Time",  72000.00, 6),
    ("EMP003", date(2021, 1, 10),  "HR",          "South", "HR Specialist",     "Part-Time",  55000.00, 2),
    ("EMP004", date(2018, 5, 30),  "Engineering", "North", "Senior Dev",        "Full-Time", 120000.00, 8),
    ("EMP005", date(2022, 9, 5),   "Finance",     "West",  "Accountant",        "Full-Time",  80000.00, 3),
    ("EMP006", date(2020, 11, 18), "Engineering", "East",  "ML Engineer",       "Full-Time", 110000.00, 5),
    ("EMP007", date(2017, 4, 12),  "Sales",       "South", "Sales Manager",     "Full-Time",  90000.00, 9),
    ("EMP008", date(2023, 2, 28),  "Marketing",   "North", "Content Writer",    "Part-Time",  48000.00, 1),
    ("EMP009", date(2021, 6, 14),  "Finance",     "West",  "Financial Analyst", "Full-Time",  85000.00, 4),
    ("EMP010", date(2019, 8, 20),  "Engineering", "East",  "DevOps Engineer",   "Full-Time", 105000.00, 7),
]

employees_df = spark.createDataFrame(employees_data, schema=employees_schema)

show(employees_df)
employees_df.printSchema()

In [ ]:
# -----------------------------------------------
# 1. Average Salary by Department
# -----------------------------------------------
print("=== Average Salary by Department ===")
show(
    employees_df.groupBy("Department")
    .agg(F.round(F.avg("Salary"), 2).alias("Avg_Salary"))
    .orderBy(F.desc("Avg_Salary"))
)

# -----------------------------------------------
# 2. Headcount by Region
# -----------------------------------------------
print("=== Headcount by Region ===")
show(
    employees_df.groupBy("Region")
    .agg(F.count("EmployeeID").alias("Total_Employees"))
    .orderBy(F.desc("Total_Employees"))
)

# -----------------------------------------------
# 3. Total Salary Cost by Department and Job Category
# -----------------------------------------------
print("=== Total Salary Cost by Department & Category ===")
show(
    employees_df.groupBy("Department", "Category")
    .agg(
        F.round(F.sum("Salary"), 2).alias("Total_Salary"),
        F.count("EmployeeID").alias("Headcount"),
    )
    .orderBy("Department", "Category")
)

# -----------------------------------------------
# 4. Max and Min Salary by Region
# -----------------------------------------------
print("=== Max & Min Salary by Region ===")
show(
    employees_df.groupBy("Region")
    .agg(
        F.max("Salary").alias("Max_Salary"),
        F.min("Salary").alias("Min_Salary"),
        F.round(F.avg("Salary"), 2).alias("Avg_Salary"),
    )
    .orderBy("Region")
)

# -----------------------------------------------
# 5. Average Years of Experience by Job Title
# -----------------------------------------------
print("=== Avg Experience by Job Title ===")
show(
    employees_df.groupBy("JobTitle")
    .agg(
        F.round(F.avg("YearsExperience"), 1).alias("Avg_Experience"),
        F.round(F.avg("Salary"), 2).alias("Avg_Salary"),
    )
    .orderBy(F.desc("Avg_Experience"))
)

# -----------------------------------------------
# 6. Filter: High Earners (Salary > 90000)
# -----------------------------------------------
print("=== High Earners (Salary > 90,000) ===")
show(
    employees_df.filter(F.col("Salary") > 90000)
    .select("EmployeeID", "JobTitle", "Department", "Salary")
    .orderBy(F.desc("Salary"))
)